# BJ-DATA-2025-SETTLEMENT-BACKFILL-001 — amendment v2

Executed, source-bound audit of the frozen 2025 settlement denominator.


In [1]:
from pathlib import Path
import json
import os
import sys
import pandas as pd
ROOT = Path(os.environ.get('BJ396_REPO_ROOT', Path.cwd())).resolve()
sys.path.insert(0, str(ROOT))
from src.bj396_settlement import verify_package
PACKAGE = ROOT / 'results/bj396_2025_settlement_backfill_v2'
assert PACKAGE.is_dir(), PACKAGE


In [2]:
verification = verify_package(PACKAGE)
verification


{'status': 'PASS', 'contract': 'BJ-DATA-2025-SETTLEMENT-BACKFILL-001/v2', 'disposition': 'READY_FOR_394_FULL_2025_REPLAY', 'files': 83, 'settlement_records': 16117, 'excluded_records': 120, 'issues': 57}

In [3]:
decision = json.loads((PACKAGE / 'decision.json').read_text(encoding='utf-8'))
residuals = pd.read_csv(PACKAGE / 'residual_disposition_v2.csv')
probes = pd.read_csv(PACKAGE / 'nowscore_probe_ledger.csv')
issue_ledger = pd.read_csv(PACKAGE / 'issue_ledger.csv')
source_probes = pd.read_csv(PACKAGE / 'source_probe_receipts_v2.csv')
print(json.dumps(decision, indent=2))
print(residuals.groupby('exclusion_reason').size())
print(source_probes[['source_id', 'exact_23_key_hit_count', 'native_winning_sp_capability', 'admissible_use']].to_string(index=False))
assert len(probes) == 23
assert len(issue_ledger) == 57 and issue_ledger['downstream_eligible'].all()
assert source_probes['source_id'].nunique() == 5


{
  "contract": "BJ-DATA-2025-SETTLEMENT-BACKFILL-001/v2",
  "disposition": "READY_FOR_394_FULL_2025_REPLAY",
  "downstream_eligible_issue_count": 57,
  "excluded_fixture_count": 120,
  "first_issue": "25011",
  "included_issue_count": 57,
  "issues_with_admissible_fixture_exclusions": 36,
  "last_issue": "25125",
  "maximum_settlement_availability_timestamp": "2026-08-25T12:27:17+00:00",
  "never_settled_official_no_sp_count": 18,
  "nowscore_probed_fixture_count": 23,
  "postponed_or_cancelled_fixture_count": 102,
  "settlement_record_count": 16117,
  "source_row_count": 16237,
  "strategy_execution_authorized": false,
  "target_issue_count": 57,
  "unresolved_native_settlement_count": 0
}
exclusion_reason
NEVER_SETTLED_OFFICIAL_NO_SP     18
POSTPONED_OR_CANCELLED          102
               source_id  exact_23_key_hit_count        native_winning_sp_capability                                admissible_use
            500_kaijiang                      23           NO_NATIVE_SP_FOR_EXA

All 57 census issues are explicitly downstream eligible: 36 retain admissibly excluded fixtures, recorded only in `residual_disposition_v2.csv`; no whole issue is silently dropped. The five-row source-probe receipt captures every declared lane and exact-23-key readback. All 23 original no-SP rows have an immutable Nowscore issue/slot probe. Five are directly provider-marked 取消 and 18 are finished with an outcome but remain excluded because the official 500 row has no native winning SP. No final score was converted to SP, and `strategy_execution_authorized` remains false.
